In [ ]:
# 安裝正確命名的新版套件與相容的 Torch
!pip install -qU pinecone sentence-transformers pypdf google-generativeai torch==2.11.0

In [ ]:
import os
import pypdf
import torch
import time
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from pinecone import Pinecone, ServerlessSpec
from google.colab import userdata
import google.generativeai as genai

# --- 1. 環境設定 ---
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))
pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
llm = genai.GenerativeModel('gemini-2.5-flash')

# --- 2. 讀取 PDF 與【優化切割策略】 (對應方式 1) ---
pdf_path = "Softwarized_Data-Driven_Architecture_for_Edge_Computing_IIoT_Environments_A_Proof_of_Concept.pdf"
reader = pypdf.PdfReader(pdf_path)
full_text = "".join([p.extract_text() for p in reader.pages if p.extract_text()])

# 方式 1：大幅增加 Overlap (150)，確保「Random Forest」與「92.3%」出現在同一個區塊
# Chunk 500, Overlap 150 -> 每個區塊有 30% 內容與前後重疊
chunk_size = 500
overlap = 150
text_chunks = [full_text[i:i+chunk_size] for i in range(0, len(full_text), chunk_size - overlap)]

# --- 3. 微調模型 (保持 RoBERTa 領域強化) ---
model = SentenceTransformer('all-distilroberta-v1')
train_examples = [InputExample(texts=[text_chunks[i], text_chunks[i+1]]) for i in range(len(text_chunks)-1)]
train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=8)
train_loss = losses.MultipleNegativesRankingLoss(model)
model.fit(train_objectives=[(train_dataloader, train_loss)], epochs=3, warmup_steps=10)

# --- 4. 存入 Pinecone +【Metadata 標籤化】 (對應方式 2, 4) ---
index_name = "iiot-advanced-rag"
if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name, dimension=768, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index(index_name)

print("正在進行 Metadata 標籤化與向量上傳...")
vectors_to_upsert = []

for i, chunk in enumerate(text_chunks):
    # 方式 4：簡單標籤化 - 檢查內容是否包含關鍵特徵
    category = "General"
    if any(kw in chunk.lower() for kw in ["result", "precision", "recall", "accuracy", "performance"]):
        category = "Experiment_Results" # 標註為實驗結果
    elif any(kw in chunk.lower() for kw in ["architecture", "cloud", "edge", "factory"]):
        category = "Architecture_Design"

    # 方式 2：父子區塊邏輯 (保留上下文)
    start_idx = max(0, i-1)
    end_idx = min(len(text_chunks), i+2)
    parent_text = "\n---\n".join(text_chunks[start_idx:end_idx])

    emb = model.encode(chunk).tolist()
    vectors_to_upsert.append({
        "id": f"c-{i}",
        "values": emb,
        "metadata": {
            "text": chunk,
            "parent_text": parent_text,
            "category": category  # 存入標籤
        }
    })

index.upsert(vectors=vectors_to_upsert)
print("知識庫建置完成。")

# --- 5. 互動檢索與【混合檢索思維】 (對應方式 3) ---
def safe_call(prompt, config=None):
    time.sleep(2)
    return llm.generate_content(prompt, generation_config=config).text.strip()

while True:
    question_zh = input("\n請輸入您的問題：").strip()
    if question_zh.lower() in ['exit', 'quit']: break

    # A. 翻譯
    question_en = safe_call(f"Translate to academic English: {question_zh}")

    # B. 檢索
    query_vec = model.encode(question_en).tolist()

    # 方式 3：擴大 Top-K (增加到 20) 來彌補純語義檢索的不足
    # 並在 Prompt 指引 LLM 優先尋找 Experiment_Results 標籤的內容
    results = index.query(vector=query_vec, top_k=20, include_metadata=True)

    # 提取並去重
    context_list = []
    seen = set()
    for res in results['matches']:
        p_text = res['metadata']['parent_text']
        if p_text not in seen:
            # 顯示標籤輔助 AI 判斷
            cat = res['metadata'].get('category', 'General')
            context_list.append(f"[{cat}]: {p_text}")
            seen.add(p_text)

    context = "\n\n".join(context_list)

    # C. 生成回答
    qa_prompt = f"""
    你是一位專業研究助理。請根據提供文獻回答問題。


    【文獻內容】：
    {context}

    【問題】：{question_zh}
    """

    answer = safe_call(qa_prompt)
    print(f"回答：\n{answer}")